# Сравнение LSTM, Transformer и Mamba: оценка и итоги



##  Загрузка заранее обученных моделей

In [11]:
import json
import math
import random
import time
from typing import Callable, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from einops import rearrange

from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cpu'


Это те же классы, что и в обучающих ноутбуках (LSTM, Transformer, Mamba), они нужны здесь только для того, чтобы загрузить веса из сохранённых моделей. 


In [12]:
class CharLSTMLM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_size=256, num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hidden_size, num_layers=num_layers, batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, state=None):
        emb = self.embedding(x)
        out, state = self.lstm(emb, state)
        out = self.dropout(out)
        return self.fc(out), state


In [13]:
class CharTransformerLM(nn.Module):
    def __init__(self, vocab_size, hidden_dim=128, num_heads=4, num_layers=2, ff_dim=256,
                 dropout=0.1, max_len=512, pad_idx=0):
        super().__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(vocab_size, hidden_dim, padding_idx=pad_idx)
        self.positional_encoding = nn.Parameter(torch.zeros(1, max_len, hidden_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        causal_mask = torch.triu(
            torch.full((seq_len, seq_len), float("-inf"), device=x.device), diagonal=1
        )
        emb = self.embedding(x) + self.positional_encoding[:, :seq_len, :]
        out = self.transformer_encoder(emb, mask=causal_mask)
        return self.fc_out(out)


In [14]:
class PScan(Function):
    @staticmethod
    def forward(ctx, A_inp, X_inp):
        A, X = A_inp.clone(), X_inp.clone()
        A, X = rearrange(A, "l b d s -> b d l s"), rearrange(X, "l b d s -> b d l s")
        A_orig = A.clone()
        PScan._forward(A, X)
        ctx.save_for_backward(A_orig, X)
        return rearrange(X, "b d l s -> b l d s")

    @staticmethod
    def backward(ctx, grad_inp: Tensor) -> Tuple[Tensor, Tensor]:
        A, X = ctx.saved_tensors
        A = torch.cat((A[:, :, :1], A[:, :, 1:].flip(2)), dim=2)
        grad_out = rearrange(grad_inp, "b l d s -> b d l s")
        grad_out = grad_out.flip(2)
        PScan._forward(A, grad_out)
        grad_out = grad_out.flip(2)
        Q = torch.zeros_like(X)
        Q[:, :, 1:].add_(X[:, :, :-1] * grad_out[:, :, 1:])
        return rearrange(Q, "b d l s -> l b d s"), rearrange(grad_out, "b d l s -> l b d s")

    @staticmethod
    def _forward(A: Tensor, X: Tensor) -> None:
        b, d, l, s = A.shape
        num_steps = int(math.log2(l))
        Av, Xv = A, X
        for _ in range(num_steps):
            T = Xv.size(2)
            Av, Xv = Av[:, :, :T].reshape(b, d, T // 2, 2, -1), Xv[:, :, :T].reshape(b, d, T // 2, 2, -1)
            Xv[:, :, :, 1].add_(Av[:, :, :, 1].mul(Xv[:, :, :, 0]))
            Av[:, :, :, 1].mul_(Av[:, :, :, 0])
            Av, Xv = Av[:, :, :, 1], Xv[:, :, :, 1]
        for k in range(num_steps - 1, -1, -1):
            Av, Xv = A[:, :, 2**k - 1: l: 2**k], X[:, :, 2**k - 1: l: 2**k]
            T = 2 * (Xv.size(2) // 2)
            if T < Xv.size(2):
                Xv[:, :, -1].add_(Av[:, :, -1].mul(Xv[:, :, -2]))
                Av[:, :, -1].mul_(Av[:, :, -2])
            Av, Xv = Av[:, :, :T].reshape(b, d, T // 2, 2, -1), Xv[:, :, :T].reshape(b, d, T // 2, 2, -1)
            Xv[:, :, 1:, 0].add_(Av[:, :, 1:, 0].mul(Xv[:, :, :-1, 1]))
            Av[:, :, 1:, 0].mul_(Av[:, :, :-1, 1])


pscan: Callable[[Tensor, Tensor], Tensor] = PScan.apply


class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=8, d_conv=4, expand=2, dt_rank=None):
        super().__init__()
        self.d_inner = expand * d_model
        self.d_state = d_state
        self.dt_rank = dt_rank or max(d_model // 16, 1)

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                 groups=self.d_inner, padding=d_conv - 1, bias=True)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * self.d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    @staticmethod
    def _next_pow2(n):
        return 1 if n <= 1 else 2 ** math.ceil(math.log2(n))

    def forward(self, x):
        b, l, d = x.shape
        x_and_res = self.in_proj(x)
        x_in, res = x_and_res.split([self.d_inner, self.d_inner], dim=-1)
        x_in_t = x_in.transpose(1, 2)
        x_in_t = self.conv1d(x_in_t)[:, :, :l]
        x_in = x_in_t.transpose(1, 2)
        x_in = F.silu(x_in)
        y = self.ssm(x_in)
        y = y * F.silu(res)
        return self.out_proj(y)

    def ssm(self, x):
        b, l, d_in = x.shape
        n = self.A_log.shape[1]
        A = -torch.exp(self.A_log.float())
        x_dbl = self.x_proj(x)
        delta, B, C = x_dbl.split([self.dt_rank, n, n], dim=-1)
        delta = F.softplus(self.dt_proj(delta))
        deltaA = torch.exp(delta.unsqueeze(-1) * A)
        deltaB_x = delta.unsqueeze(-1) * B.unsqueeze(2) * x.unsqueeze(-1)
        l_pad = self._next_pow2(l)
        if l_pad != l:
            pad = l_pad - l
            deltaA = F.pad(deltaA, (0, 0, 0, 0, 0, pad))
            deltaB_x = F.pad(deltaB_x, (0, 0, 0, 0, 0, pad))
        h = pscan(deltaA.permute(1, 0, 2, 3), deltaB_x.permute(1, 0, 2, 3))
        h = h[:, :l]
        y = (h * C.unsqueeze(2)).sum(-1)
        y = y + self.D * x
        return y


class MambaLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2, d_state=8, d_conv=4, expand=2, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.blocks = nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        h = self.embedding(x)
        for block, norm in zip(self.blocks, self.norms):
            h = h + block(norm(h))
        return self.fc_out(h)


## Загрузка моделей



In [15]:
MODEL_REGISTRY = {
    "lstm": dict(ckpt="best_lstm.pt", cls=CharLSTMLM, has_state=True),
    "transformer": dict(ckpt="best_transformer.pt", cls=CharTransformerLM, has_state=False),
    "mamba": dict(ckpt="best_mamba.pt", cls=MambaLM, has_state=False),
}

loaded = {}
for name, spec in MODEL_REGISTRY.items():
    ckpt = torch.load(spec["ckpt"], map_location=device, weights_only=False)
    model = spec["cls"](ckpt["vocab_size"], pad_idx=ckpt["char2idx"]["<pad>"], **ckpt["config"]).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    loaded[name] = dict(
        model=model, char2idx=ckpt["char2idx"], idx2char=ckpt["idx2char"],
        vocab_size=ckpt["vocab_size"], has_state=spec["has_state"],
    )
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name}: загружен, параметров={n_params:,}")

assert len(loaded) > 0, "Не загружено ни одной модели, проверьте пути"


def forward_logits(name, x):
    model = loaded[name]["model"]
    if loaded[name]["has_state"]:
        logits, _ = model(x)
    else:
        logits = model(x)
    return logits


lstm: загружен, параметров=957,790
transformer: загружен, параметров=340,574
mamba: загружен, параметров=71,518


## Расчёт Perplexity

Чтобы Perplexity была сравнима между моделями, считаем её здесь заново

In [16]:
dataset = load_dataset("inkoziev/ArsPoetica")
ds = dataset["train"]

MAX_CHARS = 400
N_COMMON_VAL = 200  

def get_text(example):
    return example["accentuation"][:MAX_CHARS]

all_texts_full = [get_text(ex) for ex in ds]
rng = random.Random(SEED)
shuffled = all_texts_full.copy()
rng.shuffle(shuffled)
COMMON_VAL_TEXTS = shuffled[:N_COMMON_VAL]
print(f"Общий валидационный набор: {len(COMMON_VAL_TEXTS)} стихотворений")


def encode_with(char2idx, text):
    ids = [char2idx["<bos>"]]
    for c in text:
        if c in char2idx:
            ids.append(char2idx[c])
    ids.append(char2idx["<eos>"])
    return ids

@torch.no_grad()
def compute_perplexity(name, texts):
    info = loaded[name]
    char2idx = info["char2idx"]
    pad_idx = char2idx["<pad>"]
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx, reduction="sum")
    total_loss, total_tokens = 0.0, 0
    for t in tqdm(texts, desc=f"perplexity[{name}]", leave=False):
        ids = encode_with(char2idx, t)
        if len(ids) < 2:
            continue
        x = torch.tensor([ids[:-1]], device=device)
        y = torch.tensor([ids[1:]], device=device)
        logits = forward_logits(name, x)
        loss = criterion(logits.reshape(-1, info["vocab_size"]), y.reshape(-1))
        total_loss += loss.item()
        total_tokens += y.numel()
    avg_loss = total_loss / max(total_tokens, 1)
    return avg_loss, math.exp(avg_loss)

perplexity_results = {}
for name in loaded:
    avg_loss, ppl = compute_perplexity(name, COMMON_VAL_TEXTS)
    perplexity_results[name] = ppl
    print(f"{name}: avg_loss={avg_loss:.3f}  perplexity={ppl:.2f}")


Общий валидационный набор: 200 стихотворений


perplexity[lstm]:   0%|          | 0/200 [00:00<?, ?it/s]

lstm: avg_loss=1.281  perplexity=3.60


perplexity[transformer]:   0%|          | 0/200 [00:00<?, ?it/s]

transformer: avg_loss=1.705  perplexity=5.50


perplexity[mamba]:   0%|          | 0/200 [00:00<?, ?it/s]

mamba: avg_loss=1.611  perplexity=5.01


## Загрузка готовых стихотворений


In [ ]:
TARGET_N_QUATRAINS = 30

QUATRAIN_JSON_PATHS = {
    "lstm": "generated_quatrains.json",
    "transformer": "generated_quatrains_transformer.json",
    "mamba": "generated_quatrains_mamba.json",
}


def extract_quatrains(text, n_lines=4):
    lines = [l for l in text.split("\n") if l.strip()]
    quatrains = []
    for i in range(0, len(lines) - n_lines + 1, n_lines):
        chunk = lines[i:i + n_lines]
        if len(chunk) == n_lines:
            quatrains.append("\n".join(chunk))
    return quatrains


@torch.no_grad()
def generate_poem(name, max_len=300, temperature=0.8, top_k=30):
    info = loaded[name]
    char2idx, idx2char = info["char2idx"], info["idx2char"]
    pad_idx, bos_idx, eos_idx = char2idx["<pad>"], char2idx["<bos>"], char2idx["<eos>"]
    seq = [bos_idx]
    for _ in range(max_len):
        x = torch.tensor([seq], device=device)
        logits = forward_logits(name, x)[0, -1] / temperature
        logits[pad_idx] = -float("inf")
        logits[bos_idx] = -float("inf")
        k = min(top_k, logits.size(-1))
        topk_vals, topk_idx = torch.topk(logits, k)
        probs = F.softmax(topk_vals, dim=-1)
        choice = topk_idx[torch.multinomial(probs, 1)].item()
        if choice == eos_idx:
            break
        seq.append(choice)
    return "".join(idx2char[i] for i in seq[1:])


quatrains_by_model = {}
for name in loaded:
    qs = []
    json_path = QUATRAIN_JSON_PATHS.get(name)
    if json_path and os.path.exists(json_path):
        qs = json.load(open(json_path, encoding="utf-8"))
    quatrains_by_model[name] = qs[:TARGET_N_QUATRAINS]
